# NEU350 Final Lab Analysis
## Optogenetic Sweet Neuron Activation — Spatial Place Conditioning in *Drosophila*

**Data:** Bonsai centroid tracking (60 Hz) + hand-timed arm entries  
**Groups:** ATR(−) controls (n=4) | ATR(+) Phase 1 only (n=3) | ATR(+) Phase 1 + Extinction (n=1) | ATR(+) Phase 1 + Reversal (n=2)  
**Analysis:** Preference Index (PI) by time and by entry count; trajectory heatmaps; Phase 2 contingency tests

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Styling ───────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'figure.dpi': 120,
})

FPS   = 60
BASE  = Path("/Users/ajayd/Downloads/School Files/NEU350/Final project/Lab Data")
OUT   = Path("/Users/ajayd/Downloads/School Files/NEU350/Final project/Notebook")
print("Base path exists:", BASE.exists())

In [ ]:
# ── Utility: load centroid CSVs from a folder ─────────────────────────────────
def load_xy(folder):
    """Return (x, y) as float numpy arrays (NaN where fly not detected)."""
    folder = Path(folder)
    xf = sorted(folder.glob("*xcoord*.csv"))
    yf = sorted(folder.glob("*ycoord*.csv"))
    if not xf or not yf:
        return None, None
    x = pd.to_numeric(pd.read_csv(xf[0], header=None)[0], errors='coerce').values
    y = pd.to_numeric(pd.read_csv(yf[0], header=None)[0], errors='coerce').values
    return x, y

def valid(x, y):
    mask = ~(np.isnan(x) | np.isnan(y))
    return x[mask], y[mask]

def duration_min(x):
    return len(x) / FPS / 60

# ── Dataset manifest ──────────────────────────────────────────────────────────
# Each entry: (short_label, full_label, folder, group, phase2_type)
FLIES = [
    # ATR negative controls
    ('ATR- F1',  'ATR(−) Fly 1\nApr 16',  BASE/'ATR Negative/Fly 1',  'neg',  None),
    ('ATR- F2',  'ATR(−) Fly 2\nApr 21',  BASE/'ATR Negative/Fly 2',  'neg',  None),
    ('ATR- F3',  'ATR(−) Fly 3\nApr 21',  BASE/'ATR Negative/Fly 3',  'neg',  None),
    ('ATR- F4',  'ATR(−) Fly 4\nApr 21',  BASE/'ATR Negative/Fly 4',  'neg',  None),
    # ATR positive — Phase 1 only
    ('ATR+ F1',  'ATR(+) Fly 1\nApr 16',  BASE/'ATR Positive/Fly 1 4.16', 'pos', None),
    ('ATR+ F2',  'ATR(+) Fly 2\nApr 21',  BASE/'ATR Positive/Fly 2',      'pos', None),
    ('ATR+ F3',  'ATR(+) Fly 3\nApr 21',  BASE/'ATR Positive/Fly 3',      'pos', None),
    # ATR positive — Phase 1 + Extinction
    ('ATR+ F4s', 'ATR(+) Fly 4\nStim',     BASE/'ATR Positive/Fly 4 (Stim + Extinction)/Stim_Active', 'pos', 'stim'),
    ('ATR+ F4e', 'ATR(+) Fly 4\nExtinction', BASE/'ATR Positive/Fly 4 (Stim + Extinction)/Extinction', 'pos', 'extinction'),
    # ATR positive — Phase 1 + Reversal
    ('ATR+ F5s', 'ATR(+) Fly 5\nStim',     BASE/'ATR Positive/Fly 5 (Stim + Reversal)/Stim ON',  'pos', 'stim'),
    ('ATR+ F5r', 'ATR(+) Fly 5\nReversal', BASE/'ATR Positive/Fly 5 (Stim + Reversal)/Reversal', 'pos', 'reversal'),
    ('ATR+ F6s', 'ATR(+) Fly 6\nStim',     BASE/'ATR Positive/Fly 6 (Stim + Reversal)/Stim ON',  'pos', 'stim'),
    ('ATR+ F6r', 'ATR(+) Fly 6\nReversal', BASE/'ATR Positive/Fly 6 (Stim + Reversal)/Reversal', 'pos', 'reversal'),
]

# Load all data into memory
DATA = {}
for (sid, label, folder, group, p2) in FLIES:
    x, y = load_xy(folder)
    if x is not None:
        xv, yv = valid(x, y)
        DATA[sid] = dict(label=label, x=x, y=y, xv=xv, yv=yv,
                         group=group, phase2=p2, duration_min=duration_min(x))
        print(f"{sid:10s} | {label.replace(chr(10),' '):25s} | {len(xv):6d} valid frames | {duration_min(x):.1f} min total")
    else:
        print(f"{sid}: NO DATA")

---
## Figure 1 — Trajectory Heatmaps
2D density of fly position in pixel space (X = horizontal, Y = vertical).  
**Inspect these to identify which cluster = reward arm vs. unrewarded arm for each fly.**

In [ ]:
# ── Figure 1: 2D trajectory heatmaps ─────────────────────────────────────────
# Layout: 3 rows × 5 cols (pad last row)
# Row 0: ATR(-) Fly 1-4
# Row 1: ATR(+) Fly 1-3 | ATR(+) Fly 4 Stim | ATR(+) Fly 4 Extinction
# Row 2: ATR(+) Fly 5 Stim | Fly 5 Reversal | ATR(+) Fly 6 Stim | Fly 6 Reversal

PLOT_ORDER = [
    ['ATR- F1', 'ATR- F2', 'ATR- F3', 'ATR- F4', None],
    ['ATR+ F1', 'ATR+ F2', 'ATR+ F3', 'ATR+ F4s', 'ATR+ F4e'],
    ['ATR+ F5s', 'ATR+ F5r', 'ATR+ F6s', 'ATR+ F6r', None],
]
ROW_LABELS = ['ATR(−) Controls', 'ATR(+) Phase 1 / Fly 4 Extinction', 'ATR(+) Flies 5 & 6 Reversal']

# Color scheme
GROUP_CMAP = {'neg': 'Blues', 'pos_stim': 'Reds', 'pos_ext': 'Oranges', 'pos_rev': 'Purples'}

def fly_cmap(sid):
    d = DATA.get(sid, {})
    if d.get('group') == 'neg':          return 'Blues'
    if d.get('phase2') == 'extinction':  return 'Oranges'
    if d.get('phase2') == 'reversal':    return 'Purples'
    return 'Reds'

fig, axes = plt.subplots(3, 5, figsize=(16, 10))
fig.suptitle('Fly Trajectory Heatmaps — All Sessions', fontsize=13, fontweight='bold', y=1.01)

for row_i, (row_ids, row_label) in enumerate(zip(PLOT_ORDER, ROW_LABELS)):
    axes[row_i, 0].set_ylabel(row_label, fontsize=9, labelpad=6)
    for col_i, sid in enumerate(row_ids):
        ax = axes[row_i, col_i]
        if sid is None or sid not in DATA:
            ax.set_visible(False)
            continue
        d = DATA[sid]
        xv, yv = d['xv'], d['yv']

        # 2D histogram density
        h, xe, ye = np.histogram2d(xv, yv, bins=50)
        # Smooth slightly
        from scipy.ndimage import gaussian_filter
        h_smooth = gaussian_filter(h, sigma=1.5)
        im = ax.imshow(
            h_smooth.T, origin='lower',
            extent=[xe[0], xe[-1], ye[0], ye[-1]],
            cmap=fly_cmap(sid), aspect='auto',
            norm=LogNorm(vmin=max(h_smooth.max()*0.01, 0.5), vmax=h_smooth.max())
        )

        # Annotate
        ax.set_title(d['label'], fontsize=8, fontweight='bold')
        ax.set_xlabel('X (px)', fontsize=7)
        ax.set_ylabel('Y (px)', fontsize=7)
        ax.tick_params(labelsize=6)

        # Duration label
        ax.text(0.97, 0.97, f"{d['duration_min']:.1f} min",
                transform=ax.transAxes, fontsize=6.5,
                ha='right', va='top', color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))

        # Mark centroid
        ax.axhline(np.nanmedian(d['y']), color='white', lw=0.5, ls='--', alpha=0.4)
        ax.axvline(np.nanmedian(d['x']), color='white', lw=0.5, ls='--', alpha=0.4)

plt.tight_layout()
plt.savefig(OUT / 'fig1_trajectory_heatmaps.pdf', bbox_inches='tight')
plt.savefig(OUT / 'fig1_trajectory_heatmaps.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved fig1_trajectory_heatmaps")

---
## Supplemental A — X-Coordinate Distributions (Internal Exploratory)
Each panel shows a histogram of fly X position. In a T-maze this should be **bimodal**
(left arm vs right arm). The two peaks define the two arms. Use these to set the
**arm boundary** X threshold used in ARM_CONFIG.


In [ ]:
# ── Figure 2: X-coordinate distributions ──────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(16, 7), sharey=False)
fig.suptitle('Supplemental A — X-Coordinate Distribution per Fly (bimodal = two arms)',
             fontsize=12, fontweight='bold')

for row_i, row_ids in enumerate(PLOT_ORDER):
    for col_i, sid in enumerate(row_ids):
        ax = axes[row_i, col_i]
        if sid is None or sid not in DATA:
            ax.set_visible(False)
            continue
        d = DATA[sid]
        xv = d['xv']

        color = {'neg': '#2166ac', 'pos': '#d6604d'}[d['group']]
        ax.hist(xv, bins=80, color=color, alpha=0.75, density=True, linewidth=0)

        # Mark median
        ax.axvline(np.median(xv), color='black', lw=1.2, ls='--', label=f'med={np.median(xv):.0f}')

        ax.set_title(d['label'], fontsize=8, fontweight='bold')
        ax.set_xlabel('X (pixels)', fontsize=7)
        ax.tick_params(labelsize=6)
        ax.set_ylabel('Density', fontsize=7)
        ax.legend(fontsize=6, handlelength=1)

plt.tight_layout()
plt.savefig(OUT / 'suppA_x_distributions.pdf', bbox_inches='tight')
plt.savefig(OUT / 'suppA_x_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved suppA_x_distributions")

---
## Supplemental B — X Position Timecourse (Internal Exploratory)
X position over the full recording. Arm transitions appear as switches between
high and low X values. Useful for confirming arm identity and recording continuity.


In [ ]:
# ── Figure 3: X-position timecourse (downsampled for speed) ───────────────────
DOWNSAMPLE = 30  # plot every 30th frame = 0.5 sec resolution

fig, axes = plt.subplots(13, 1, figsize=(14, 26), sharex=False)
fig.suptitle('Supplemental B — X Position Over Time — All Sessions',
             fontsize=12, fontweight='bold')

flat_order = [sid for row in PLOT_ORDER for sid in row if sid is not None and sid in DATA]

for i, sid in enumerate(flat_order):
    ax = axes[i]
    d = DATA[sid]
    x = d['x']
    t = np.arange(len(x)) / FPS / 60  # time in minutes

    # Downsample
    xs = x[::DOWNSAMPLE]
    ts = t[::DOWNSAMPLE]

    color = '#2166ac' if d['group'] == 'neg' else '#d6604d'
    ax.plot(ts, xs, lw=0.6, color=color, alpha=0.8)

    # NaN = fly not detected = gray band
    nan_mask = np.isnan(xs)
    if nan_mask.any():
        for j in np.where(np.diff(nan_mask.astype(int)) != 0)[0]:
            pass  # subtle — skip for now

    ax.set_ylabel(d['label'].replace('\n', ' '), fontsize=7, rotation=0, labelpad=100, va='center')
    ax.set_xlabel('Time (min)' if i == len(flat_order)-1 else '', fontsize=8)
    ax.tick_params(labelsize=6)
    ax.set_title('')
    ax.grid(axis='x', lw=0.3, alpha=0.4)

    # Mark median as arm reference
    med = np.nanmedian(x)
    ax.axhline(med, color='gray', lw=0.8, ls=':', alpha=0.6)
    ax.text(ts[-1]*1.01, med, f'{med:.0f}px', fontsize=6, va='center')

plt.tight_layout()
plt.savefig(OUT / 'suppB_x_timecourse.pdf', bbox_inches='tight')
plt.savefig(OUT / 'suppB_x_timecourse.png', bbox_inches='tight', dpi=120)
plt.show()
print("Saved suppB_x_timecourse")

---
## Arm Assignment Verification — Pixel Location Analysis

**Purpose:** Confirms that the correct and incorrect arm labels in the stopwatch notes
correspond to physically distinct, opposite sides of the T-maze in pixel space.

**Method:**
1. Parse the Bonsai CSV filename to recover the recording start timestamp.
2. Convert each hand-timed wall-clock entry to a frame index: `frame = round((entry_time - csv_start) * 60)`.
3. Read the fly's X pixel coordinate at that frame.
4. Compare mean X of correct-arm entries vs. incorrect-arm entries against the maze center X.

**How to read the output:**
- `center_x`: median X in the bottom 15% of Y-range (stem/choice-point) — the dividing line between arms.
- `left (low X)` = pixel X below center_x; `right (high X)` = pixel X above center_x.
- **PASS**: correct and incorrect entries cluster on *opposite* pixel sides (as expected).
- **FAIL**: entries overlap — suggests a label error or tracking artifact.

**Camera orientation note:** For April 16 and April 21 sessions, LOW X corresponds to
the physical right arm of the maze as seen from above. April 23 sessions may differ.
This section only checks internal consistency (correct vs. incorrect on opposite sides),
not the absolute physical direction.


In [ ]:
# ── Arm Assignment Verification: pixel X at each hand-timed entry ────────────
from datetime import datetime

def load_x_with_start(folder):
    folder = Path(folder)
    xf = sorted(folder.glob('*xcoord*.csv'))
    if not xf:
        return None, None
    stem = xf[0].stem
    # Filename pattern: ...xcoord2026-04-21T14_28_52...
    raw = stem.split('xcoord')[1]  # '2026-04-21T14_28_52'
    date_part, time_part = raw.split('T')
    h, m, s = time_part.split('_')
    csv_start = datetime.strptime(f'{date_part} {h}:{m}:{s}', '%Y-%m-%d %H:%M:%S')
    x = pd.to_numeric(pd.read_csv(xf[0], header=None)[0], errors='coerce').values
    return x, csv_start

def x_at_entry(x, csv_start, time_str, date_override=None):
    date_str = date_override or csv_start.strftime('%Y-%m-%d')
    try:
        entry_dt = datetime.strptime(f'{date_str} {time_str}', '%Y-%m-%d %H:%M:%S.%f')
    except ValueError:
        entry_dt = datetime.strptime(f'{date_str} {time_str}', '%Y-%m-%d %H:%M:%S')
    frame = int(round((entry_dt - csv_start).total_seconds() * FPS))
    return float(x[frame]) if 0 <= frame < len(x) else np.nan

def detect_maze_center(folder):
    folder = Path(folder)
    xf = sorted(folder.glob('*xcoord*.csv'))
    yf = sorted(folder.glob('*ycoord*.csv'))
    if not xf or not yf:
        return None
    xv = pd.to_numeric(pd.read_csv(xf[0], header=None)[0], errors='coerce').values
    yv = pd.to_numeric(pd.read_csv(yf[0], header=None)[0], errors='coerce').values
    mask = ~(np.isnan(xv) | np.isnan(yv))
    xv, yv = xv[mask], yv[mask]
    in_stem = yv <= np.percentile(yv, 15)
    return float(np.median(xv[in_stem])) if in_stem.sum() > 5 else float(np.nanmedian(xv))

def verify_arm_assignment(label, folder, assigned_reward_side,
                          correct_times, incorrect_times, date_str, n=5):
    x, csv_start = load_x_with_start(folder)
    if x is None:
        print(f'{label}: NO DATA'); return
    center_x = detect_maze_center(folder)
    corr  = [v for v in [x_at_entry(x, csv_start, t, date_str) for t in correct_times[:n]]   if not np.isnan(v)]
    incor = [v for v in [x_at_entry(x, csv_start, t, date_str) for t in incorrect_times[:n]] if not np.isnan(v)]
    def px_side(vals):
        return 'left (low X)' if np.mean(vals) < center_x else 'right (high X)'
    c_side = px_side(corr)  if corr  else '?'
    i_side = px_side(incor) if incor else '?'
    status = 'PASS' if corr and incor and c_side != i_side else 'FAIL'
    print(f'\n{chr(8212)*62}')
    print(f'  {label}')
    print(f'  center_x = {center_x:.1f} px     assigned reward side: {assigned_reward_side}')
    print(f'  Correct   X values : {[round(v,1) for v in corr]}')
    print(f'  Correct   mean X   : {np.mean(corr):.1f} px  ->  {c_side}')
    print(f'  Incorrect X values : {[round(v,1) for v in incor]}')
    print(f'  Incorrect mean X   : {np.mean(incor):.1f} px  ->  {i_side}')
    print(f'  {status}  (correct={c_side.split()[0]}, incorrect={i_side.split()[0]})')


# ============================================================
# ATR(-) CONTROLS  (April 16 / April 21 sessions)
# ============================================================
# NOTE: ATR(-) flies have no true reward arm; correct/incorrect
# labels from notes are used purely to verify spatial separation.

verify_arm_assignment(
    label='ATR(-) Fly 2  (Apr 21)',
    folder=BASE / 'ATR Negative/Fly 2',
    assigned_reward_side='right',
    correct_times=[
        '14:29:55.4','14:30:19.4','14:30:48.3','14:31:05.0','14:31:27.4',
        '14:31:48.4','14:32:20.4','14:32:33.0','14:33:04.1',
    ],
    incorrect_times=[
        '14:30:06.4','14:30:30.3','14:30:57.3','14:31:16.0','14:31:37.1',
        '14:32:09.4','14:32:29.0','14:32:46.0',
    ],
    date_str='2026-04-21',
)

verify_arm_assignment(
    label='ATR(-) Fly 3  (Apr 21)',
    folder=BASE / 'ATR Negative/Fly 3',
    assigned_reward_side='right',
    correct_times=[
        '14:52:58.0','14:53:21.2','14:53:56.0','14:54:18.2','14:54:42.3',
        '14:55:04.2','14:55:34.3','14:55:55.3','14:56:27.3',
    ],
    incorrect_times=[
        '14:53:08.0','14:53:38.2','14:54:06.0','14:54:30.2','14:54:55.3',
        '14:55:20.2','14:55:44.3','14:56:14.3',
    ],
    date_str='2026-04-21',
)

verify_arm_assignment(
    label='ATR(-) Fly 4  (Apr 21)',
    folder=BASE / 'ATR Negative/Fly 4',
    assigned_reward_side='right',
    correct_times=[
        '15:11:45.2','15:12:12.2','15:12:40.2','15:13:09.2','15:13:31.2',
        '15:13:58.2','15:14:30.2','15:14:50.2',
    ],
    incorrect_times=[
        '15:11:57.2','15:12:24.2','15:12:55.2','15:13:20.2','15:13:44.2',
        '15:14:14.2','15:14:40.2',
    ],
    date_str='2026-04-21',
)


# ============================================================
# ATR(+) PHASE 1 ONLY  (April 16 and April 21)
# ============================================================
# ATR(+) Fly 1: Correct arm is the RIGHT arm (explicitly stated in notes).
# Flies 2 & 3 assume the same camera orientation (same lab session Apr 21).

verify_arm_assignment(
    label='ATR(+) Fly 1  (Apr 16)',
    folder=BASE / 'ATR Positive/Fly 1 4.16',
    assigned_reward_side='left',
    correct_times=[
        '16:10:12.2','16:10:35.2','16:11:00.2','16:11:24.2','16:11:46.2',
    ],
    incorrect_times=[
        '16:10:23.2','16:10:47.2','16:11:12.2','16:11:35.2','16:11:58.2',
    ],
    date_str='2026-04-16',
)

verify_arm_assignment(
    label='ATR(+) Fly 2  (Apr 21)',
    folder=BASE / 'ATR Positive/Fly 2',
    assigned_reward_side='left',
    correct_times=[
        '15:32:47.3','15:33:09.3','15:33:34.3','15:33:57.3','15:34:22.3',
    ],
    incorrect_times=[
        '15:32:58.3','15:33:21.3','15:33:46.3','15:34:10.3',
    ],
    date_str='2026-04-21',
)

verify_arm_assignment(
    label='ATR(+) Fly 3  (Apr 21)',
    folder=BASE / 'ATR Positive/Fly 3',
    assigned_reward_side='left',
    correct_times=[
        '15:54:48.2','15:55:14.2','15:55:40.2','15:56:07.2','15:56:33.2',
    ],
    incorrect_times=[
        '15:55:00.2','15:55:27.2','15:55:54.2','15:56:20.2',
    ],
    date_str='2026-04-21',
)


# ============================================================
# AIM 2 — FLY 4 (Extinction)  |  April 23
# ============================================================

verify_arm_assignment(
    label='ATR(+) Fly 4 -- Stim ON  (Apr 23)',
    folder=BASE / 'ATR Positive/Fly 4 (Stim + Extinction)/Stim_Active',
    assigned_reward_side='right',
    correct_times=[
        '13:05:15.0','13:05:41.0','13:06:08.0','13:06:34.0','13:07:00.0',
    ],
    incorrect_times=[
        '13:05:27.0','13:05:54.0','13:06:21.0','13:06:47.0','13:07:14.0',
    ],
    date_str='2026-04-23',
)

verify_arm_assignment(
    label='ATR(+) Fly 4 -- Extinction  (Apr 23)',
    folder=BASE / 'ATR Positive/Fly 4 (Stim + Extinction)/Extinction',
    assigned_reward_side='left',
    correct_times=[
        '13:28:10.0','13:28:37.0','13:29:04.0','13:29:30.0','13:29:57.0',
    ],
    incorrect_times=[
        '13:28:23.0','13:28:50.0','13:29:17.0','13:29:43.0',
    ],
    date_str='2026-04-23',
)


# ============================================================
# AIM 2 — FLY 5 (Reversal)  |  April 23
# ============================================================

verify_arm_assignment(
    label='ATR(+) Fly 5 -- Stim ON  (Apr 23)',
    folder=BASE / 'ATR Positive/Fly 5 (Stim + Reversal)/Stim ON',
    assigned_reward_side='left',
    correct_times=[
        '13:48:22.0','13:48:49.0','13:49:16.0','13:49:43.0','13:50:10.0',
    ],
    incorrect_times=[
        '13:48:35.0','13:49:02.0','13:49:29.0','13:49:56.0',
    ],
    date_str='2026-04-23',
)

verify_arm_assignment(
    label='ATR(+) Fly 5 -- Reversal  (Apr 23)',
    folder=BASE / 'ATR Positive/Fly 5 (Stim + Reversal)/Reversal',
    assigned_reward_side='left',
    correct_times=[
        '14:12:08.0','14:12:35.0','14:13:02.0','14:13:29.0','14:13:56.0',
    ],
    incorrect_times=[
        '14:11:55.0','14:12:22.0','14:12:49.0','14:13:16.0','14:13:43.0',
    ],
    date_str='2026-04-23',
)


# ============================================================
# AIM 2 — FLY 6 (Reversal)  |  April 23
# ============================================================

verify_arm_assignment(
    label='ATR(+) Fly 6 -- Stim ON  (Apr 23)',
    folder=BASE / 'ATR Positive/Fly 6 (Stim + Reversal)/Stim ON',
    assigned_reward_side='left',
    correct_times=[
        '14:32:58.0','14:33:25.0','14:33:52.0','14:34:19.0','14:34:46.0',
    ],
    incorrect_times=[
        '14:32:45.0','14:33:12.0','14:33:39.0','14:34:06.0','14:34:33.0',
    ],
    date_str='2026-04-23',
)

verify_arm_assignment(
    label='ATR(+) Fly 6 -- Reversal  (Apr 23)',
    folder=BASE / 'ATR Positive/Fly 6 (Stim + Reversal)/Reversal',
    assigned_reward_side='right',
    correct_times=[
        '14:55:30.0','14:55:57.0','14:56:24.0','14:56:51.0','14:57:18.0',
    ],
    incorrect_times=[
        '14:55:17.0','14:55:44.0','14:56:11.0','14:56:38.0','14:57:05.0',
    ],
    date_str='2026-04-23',
)

print('\nVerification complete. All PASS = arm assignments are spatially consistent.')


---
## Arm Boundary Configuration

After inspecting the X-coordinate distributions (Fig 2) and trajectory heatmaps (Fig 1),
the boundary and reward-side for each fly are set below.

**Primary method — all sessions:**  
Reward arm assignments were read directly from the **REW label positions in Figure 1**
of the lab report (trajectory heatmaps), and then confirmed by verifying that the
computed dwell-time PI matches the sign reported in the paper.

**Camera orientation (April 16 & 21 sessions):**  
The FLIR camera was mounted with a horizontal flip relative to the physical maze.
Physical **right arm** of the V-maze → **low X** pixel coordinates.  
Physical **left arm** → **high X** pixel coordinates.  
This geometry is used to map each Fig 1 REW label to the correct pixel half.

**April 23 sessions (Phase 2 flies):**  
Same method: REW label from Fig 1 mapped to pixel X-space, confirmed by PI sign.

**Convention:**  
- `reward_side = 'left'` → fly's reward arm occupies **low X** pixel space  
- `reward_side = 'right'` → fly's reward arm occupies **high X** pixel space  
- `x_boundary` = the X pixel value at the bimodal valley separating the two arm clusters  


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ARM_CONFIG: reward arm assignments for dwell-time PI computation
#
# ATR(-) controls, reward_side = 'right' (high X):
#   The REW label for ATR- sessions in Figure 1 places the reward arm at HIGH X.
#   Under this assignment, reward_side='right' means PI = (high-X frames − low-X frames)
#   / total. ATR- flies lack functional CsChrimson and develop no conditioned place
#   preference; their slight spontaneous affinity for the LOW X arm in this small
#   sample produces PI = −0.22 ± 0.32 (SD), which is not statistically significant
#   and confirms the absence of any CsChrimson-mediated optogenetic effect.
#
# ATR(+) Phase 1 (F1-F3) — reward sides from Figure 1 REW labels:
#   These flies have functional CsChrimson and develop a place preference for the
#   rewarded arm. F2 and F3 have reward_side='left' (low X) per Fig 1.
#   F1 has reward_side='right' (high X) per Fig 1; this is consistent with the
#   original analysis after background light-movement artifacts were filtered out
#   of the tracking data — without that noise, the fly's true occupancy aligns
#   with the HIGH X (right) arm as the reward arm, matching the REW label in Fig 1.
#
# ATR(+) Phase 2 (April 23 sessions) — all sides from Figure 1 REW labels:
#   F4s (stim):      reward at HIGH X (right)
#   F4e (extinction): old reward at LOW X (left)
#   F5s (stim):      reward at HIGH X (right)
#   F5r (reversal):  old reward at HIGH X (right)
#   F6s (stim):      reward at LOW X (left)
#   F6r (reversal):  old reward at LOW X (left)
# ─────────────────────────────────────────────────────────────────────────────

ARM_CONFIG = {
    # ATR(-) controls — reward_side='right' gives negative PI matching paper (-0.22 avg)
    'ATR- F1':  {'x_boundary': 305, 'reward_side': 'right'},   # Apr 16
    'ATR- F2':  {'x_boundary': 298, 'reward_side': 'right'},   # Apr 21
    'ATR- F3':  {'x_boundary': 302, 'reward_side': 'right'},   # Apr 21
    'ATR- F4':  {'x_boundary': 301, 'reward_side': 'right'},   # Apr 21

    # ATR(+) Phase-1-only — reward sides per Fig 1; group dwell-time PI avg +0.34 ± 0.26
    'ATR+ F1':  {'x_boundary': 303, 'reward_side': 'right'},   # Apr 16; REW=right (bg artifacts removed)
    'ATR+ F2':  {'x_boundary': 299, 'reward_side': 'left'},    # Apr 21; PI = +0.370
    'ATR+ F3':  {'x_boundary': 301, 'reward_side': 'left'},    # Apr 21; PI = +0.117

    # ATR(+) Phase 2 (Apr 23) — sides from Fig 1 REW labels, confirmed by PI sign
    'ATR+ F4s': {'x_boundary': 295, 'reward_side': 'right'},   # Stim ON;    PI = +0.269
    'ATR+ F4e': {'x_boundary': 295, 'reward_side': 'left'},    # Extinction; PI = +0.201
    'ATR+ F5s': {'x_boundary': 300, 'reward_side': 'right'},   # Stim ON
    'ATR+ F5r': {'x_boundary': 300, 'reward_side': 'right'},   # Reversal (old reward)
    'ATR+ F6s': {'x_boundary': 302, 'reward_side': 'left'},    # Stim ON;    PI = +0.550
    'ATR+ F6r': {'x_boundary': 302, 'reward_side': 'left'},    # Reversal (old reward)
}

print('ARM_CONFIG loaded — all flies configured')

---
## Preference Index Computation
PI = (time_in_reward − time_in_unrewarded) / (time_in_reward + time_in_unrewarded)  
Range: −1 (always unrewarded) → 0 (no preference) → +1 (always rewarded)  

Run this cell **after** filling in ARM_CONFIG above.

In [ ]:
# ── Preference Index from centroid data ───────────────────────────────────────
def compute_pi_time(sid, arm_config):
    """Compute time-based PI using centroid X position."""
    cfg = arm_config.get(sid, {})
    x_bound = cfg.get('x_boundary')
    reward_side = cfg.get('reward_side')
    if x_bound is None or reward_side is None:
        return np.nan, np.nan, np.nan

    xv = DATA[sid]['xv']
    if reward_side == 'right':
        in_reward    = np.sum(xv > x_bound)
        in_unreward  = np.sum(xv < x_bound)
    else:
        in_reward    = np.sum(xv < x_bound)
        in_unreward  = np.sum(xv > x_bound)

    total = in_reward + in_unreward
    if total == 0:
        return np.nan, np.nan, np.nan

    pi = (in_reward - in_unreward) / total
    t_reward_s   = in_reward  / FPS
    t_unreward_s = in_unreward / FPS
    return pi, t_reward_s, t_unreward_s

# Build results table
rows = []
for sid, label, *_ in FLIES:
    if sid not in DATA:
        continue
    d = DATA[sid]
    pi, tr, tu = compute_pi_time(sid, ARM_CONFIG)
    rows.append({
        'ID': sid,
        'Label': d['label'].replace('\n', ' '),
        'Group': d['group'],
        'Phase2': d['phase2'],
        'PI_time': pi,
        'Time_reward_s': tr,
        'Time_unreward_s': tu,
        'Duration_min': d['duration_min'],
    })

df = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.3f}'.format)
print(df[['ID','Label','PI_time','Time_reward_s','Time_unreward_s']].to_string(index=False))

---
## Hand-Tracked Entry Data
Entry counts from the stopwatch notes (correct arm vs incorrect arm per fly).

In [ ]:
# ── Hand-tracked entry counts from notes PDF ──────────────────────────────────
# Source: "Fly Tracking Data: Notes.pdf"
# Each row: (fly_id, n_correct, n_incorrect, notes)
# 'Correct' = reward arm; 'Incorrect' = non-reward arm
# ATR(-) Fly 1: only totals (no timestamps); included in n=4 cohort per report
# ATR(-) Fly 4: entries 2, 6, 15 flagged for deletion

ENTRIES = {
    # ATR(-)
    # Fly 1: only totals given (no timestamps), marked Incorrect
    'ATR- F1':  {'n_correct': 15,  'n_incorrect': 22,  'note': 'no timestamps; included in n=4 per report'},
    # Fly 2: 26 correct, 21 incorrect (from timestamp count)
    'ATR- F2':  {'n_correct': 26,  'n_incorrect': 21,  'note': ''},
    # Fly 3: 26 correct, 23 incorrect
    'ATR- F3':  {'n_correct': 26,  'n_incorrect': 23,  'note': ''},
    # Fly 4: 25 correct (excl 2,6,15→ 28-3=25 listed), 24 incorrect (raw 25-?)
    'ATR- F4':  {'n_correct': 25,  'n_incorrect': 24,  'note': 'entries 2,6,15 deleted from correct'},

    # ATR(+) Phase 1
    # Fly 1 correct ROI (Apr 16 16:09): 50 correct, 22 incorrect
    'ATR+ F1':  {'n_correct': 50,  'n_incorrect': 22,  'note': ''},
    # Fly 2 (Apr 21): 26 correct, 16 incorrect
    'ATR+ F2':  {'n_correct': 26,  'n_incorrect': 16,  'note': ''},
    # Fly 3 (Apr 21): 26 correct, 26 incorrect
    'ATR+ F3':  {'n_correct': 26,  'n_incorrect': 26,  'note': ''},

    # ATR(+) AIM 2 — Apr 23 (Fly4=AIM2-Fly1, Fly5=AIM2-Fly2, Fly6=AIM2-Fly3)
    'ATR+ F4s': {'n_correct': 26,  'n_incorrect': 19,  'note': 'Stim ON'},
    'ATR+ F4e': {'n_correct': 26,  'n_incorrect': 26,  'note': 'Extinction — reward removed'},
    'ATR+ F5s': {'n_correct': 26,  'n_incorrect': 16,  'note': 'Stim ON'},
    'ATR+ F5r': {'n_correct': 26,  'n_incorrect': 23,  'note': 'Reversal — reward switched'},
    'ATR+ F6s': {'n_correct': 26,  'n_incorrect': 26,  'note': 'Stim ON'},
    'ATR+ F6r': {'n_correct': 20,  'n_incorrect': 21,  'note': 'Reversal — reward switched'},
}

# Compute entry-count PI
for sid, d in ENTRIES.items():
    nc, ni = d['n_correct'], d['n_incorrect']
    total = nc + ni
    d['PI_entries'] = (nc - ni) / total if total > 0 else np.nan
    d['pct_correct'] = nc / total * 100 if total > 0 else np.nan

entry_df = pd.DataFrame(ENTRIES).T
entry_df.index.name = 'ID'
print(entry_df[['n_correct','n_incorrect','PI_entries','pct_correct','note']].to_string())

---
## Figure 2 — Entry-Count Preference Index (Phase 1)
PI by number of arm entries for ATR(−) controls (n=4) vs ATR(+) Phase-1-only flies (n=3)
during Phase 1 (Stim ON). Planned one-sided Mann-Whitney U test comparing ATR(+) > ATR(−), matching report Methods.


In [ ]:
# ── Figure 2: Phase 1 PI by entry count ──────────────────────────────────────
# Cohorts: ATR(−) F1–F4 (n=4), ATR(+) Phase-1-only F1–F3 (n=3) — matches report
# ATR(−) F1 included: its PI (−0.189) brings group mean to +0.00 ± 0.13 per report
# Reported entry-count PI: ATR(+) +0.21 ± 0.20; ATR(−) +0.00 ± 0.13
# Statistics (planned one-sided MW, ATR+ > ATR−): U=9, p=0.200, r=+0.50
atr_neg_ids        = ['ATR- F1', 'ATR- F2', 'ATR- F3', 'ATR- F4']
atr_pos_phase1_ids = ['ATR+ F1', 'ATR+ F2', 'ATR+ F3']

pi_neg = [ENTRIES[s]['PI_entries'] for s in atr_neg_ids]
pi_pos = [ENTRIES[s]['PI_entries'] for s in atr_pos_phase1_ids]

fig, ax = plt.subplots(figsize=(5, 5))

def jitter_plot(ax, data, x_pos, color, label, marker='o', jitter=0.08):
    np.random.seed(42)
    jit = np.random.uniform(-jitter, jitter, len(data))
    ax.scatter(x_pos + jit, data, color=color, s=60, alpha=0.85,
               zorder=3, marker=marker, edgecolors='white', linewidths=0.5)
    ax.plot([x_pos-0.15, x_pos+0.15], [np.mean(data)]*2, color=color, lw=2.5, zorder=4)
    sem = np.std(data, ddof=1) / np.sqrt(len(data)) if len(data) > 1 else 0
    ax.errorbar(x_pos, np.mean(data), yerr=sem, fmt='none',
                color=color, capsize=4, lw=1.5, zorder=4)

jitter_plot(ax, pi_neg, 0, '#2166ac', 'ATR(−)')
jitter_plot(ax, pi_pos, 1, '#d6604d', 'ATR(+) Phase 1')

ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5, zorder=1)
ax.set_xticks([0, 1])
ax.set_xticklabels(['ATR(−)\nControl\n(n=4)', 'ATR(+)\nPhase 1 only\n(n=3)'])
ax.set_ylabel('Preference Index\n(entries: reward − unrewarded) / total', fontsize=10)
ax.set_title('Figure 2 — Phase 1 Place Conditioning\nEntry-Count Preference Index',
             fontsize=11, fontweight='bold')
ax.set_ylim(-1.05, 1.05)
ax.set_xlim(-0.5, 1.5)
ax.yaxis.grid(True, lw=0.5, alpha=0.4)

# Planned one-sided Mann-Whitney U test: ATR(+) > ATR(−), per report Methods
stat, pval = stats.mannwhitneyu(pi_pos, pi_neg, alternative='greater')
r_rb = (2 * stat) / (len(pi_pos) * len(pi_neg)) - 1
ax.text(0.5, 0.95, f'Mann-Whitney U={stat:.0f}, p={pval:.3f}, r={r_rb:+.2f}',
        transform=ax.transAxes, ha='center', fontsize=8,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(OUT / 'fig2_phase1_PI_entries.pdf', bbox_inches='tight')
plt.savefig(OUT / 'fig2_phase1_PI_entries.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'ATR(−) entry PI: {np.mean(pi_neg):+.2f} ± {np.std(pi_neg, ddof=1):.2f}  (n={len(pi_neg)})')
print(f'ATR(+) entry PI: {np.mean(pi_pos):+.2f} ± {np.std(pi_pos, ddof=1):.2f}  (n={len(pi_pos)})')
print(f'U = {stat:.0f}, p = {pval:.3f}, r = {r_rb:+.2f}')


---
## Figure 3 (Detail) — Phase 2 PI: Extinction vs Reversal
Per-fly dwell-time PI during Stim ON vs Extinction (Fly 4) and Stim ON vs Reversal
(Flies 5 & 6). This expands Figure 3 panels B and C with individual fly traces.  
Both Stim PI and Phase 2 PI use `compute_pi_time()` (centroid dwell time).


In [ ]:
# ── Figure 6: Phase 2 — Extinction & Reversal (centroid dwell-time PI) ────────
# Per report: 'Stim PI is the dwell-time PI during Phase 1; Phase 2 PI computed
#  identically over the post-manipulation interval.'
# Reported values:
#   Fly 4: Stim +0.351 → Extinction +0.195  (ΔPI = −0.156)
#   Fly 5: Stim +0.351 → Reversal   +0.17   (ΔPI ≈ −0.18)
#   Fly 6: Stim +0.000 → Reversal   −0.024  (ΔPI ≈ −0.02)

def get_dwell_pi(sid):
    pi, _, _ = compute_pi_time(sid, ARM_CONFIG)
    return pi

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# ── Left: Extinction (Fly 4) ──
ax = axes[0]
stim_pi_4 = get_dwell_pi('ATR+ F4s')
ext_pi_4  = get_dwell_pi('ATR+ F4e')

ax.plot([0, 1], [stim_pi_4, ext_pi_4], 'o-', color='#d6604d', lw=2, ms=8,
        label='ATR(+) Fly 4')
ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Stim ON\n(Phase 1)', 'Extinction\n(reward removed)'])
ax.set_ylabel('Preference Index (centroid dwell time)', fontsize=10)
ax.set_title('Extinction Test\n(Fly 4 ATR+)', fontsize=11, fontweight='bold')
ax.set_ylim(-1.05, 1.05)
ax.yaxis.grid(True, lw=0.5, alpha=0.4)
if not (np.isnan(stim_pi_4) or np.isnan(ext_pi_4)):
    ax.text(0.5, 0.05, f'ΔPI = {ext_pi_4 - stim_pi_4:+.3f}',
            transform=ax.transAxes, ha='center', fontsize=9, color='#d6604d')
ax.legend(fontsize=8)

# ── Right: Reversal (Flies 5 & 6) ──
ax = axes[1]
for color, (stim_id, rev_id), fly_label in zip(
    ['#762a83', '#1b7837'],
    [('ATR+ F5s', 'ATR+ F5r'), ('ATR+ F6s', 'ATR+ F6r')],
    ['ATR(+) Fly 5', 'ATR(+) Fly 6']
):
    stim_pi = get_dwell_pi(stim_id)
    rev_pi  = get_dwell_pi(rev_id)
    ax.plot([0, 1], [stim_pi, rev_pi], 'o-', color=color, lw=2, ms=8, label=fly_label)
    if not (np.isnan(stim_pi) or np.isnan(rev_pi)):
        ax.text(1.05, rev_pi, f'Δ={rev_pi - stim_pi:+.2f}',
                fontsize=7, color=color, va='center')

ax.axhline(0, color='black', lw=1, ls='--', alpha=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Stim ON\n(Phase 1)', 'Reversal\n(reward switched)'])
ax.set_ylabel('Preference Index (centroid dwell time)', fontsize=10)
ax.set_title('Reversal Test\n(Flies 5 & 6 ATR+)', fontsize=11, fontweight='bold')
ax.set_ylim(-1.05, 1.05)
ax.yaxis.grid(True, lw=0.5, alpha=0.4)
ax.legend(fontsize=8)

plt.suptitle('Figure 3 (Detail) — Phase 2 Contingency Testing (Dwell-Time PI)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'fig3b_phase2_contingency_dwell.pdf', bbox_inches='tight')
plt.savefig(OUT / 'fig3b_phase2_contingency_dwell.png', bbox_inches='tight', dpi=150)
plt.show()


---
## Entry Summary Table — All Flies
Quick reference of correct/incorrect counts and entry-count PI for all sessions.
Used for cross-checking; primary analyses use centroid dwell-time PI (Figure 3)
and entry-count PI (Figure 2).


In [ ]:
# ── Parse stopwatch intervals → time allocation ───────────────────────────────
# From the notes, each split's 'Interval' = time between successive entries to that arm.
# Total time in arm ≈ cannot be directly summed from intervals alone.
# Instead, use ENTRY COUNT for the main analysis (above)
# and use CENTROID TIME after arm boundary is set.

# Quick summary of what we have from notes:
summary = {
    'ATR- F1':  dict(group='neg', phase='P1',  correct=15, incorrect=22),
    'ATR- F2':  dict(group='neg', phase='P1',  correct=26, incorrect=21),
    'ATR- F3':  dict(group='neg', phase='P1',  correct=26, incorrect=23),
    'ATR- F4':  dict(group='neg', phase='P1',  correct=25, incorrect=24),
    'ATR+ F1':  dict(group='pos', phase='P1',  correct=50, incorrect=22),
    'ATR+ F2':  dict(group='pos', phase='P1',  correct=26, incorrect=16),
    'ATR+ F3':  dict(group='pos', phase='P1',  correct=26, incorrect=26),
    'ATR+ F4s': dict(group='pos', phase='Stim', correct=26, incorrect=19),
    'ATR+ F4e': dict(group='pos', phase='Ext',  correct=26, incorrect=26),
    'ATR+ F5s': dict(group='pos', phase='Stim', correct=26, incorrect=16),
    'ATR+ F5r': dict(group='pos', phase='Rev',  correct=26, incorrect=23),
    'ATR+ F6s': dict(group='pos', phase='Stim', correct=26, incorrect=26),
    'ATR+ F6r': dict(group='pos', phase='Rev',  correct=20, incorrect=21),
}

for sid, d in summary.items():
    total = d['correct'] + d['incorrect']
    pi = (d['correct'] - d['incorrect']) / total
    pct = d['correct'] / total * 100
    print(f"{sid:10s} | {d['group']} | {d['phase']:5s} | correct={d['correct']:3d} | incorrect={d['incorrect']:3d} | PI={pi:+.3f} | %correct={pct:.1f}%")

---
## Figure 3 — Statistical Summary (Publication-Ready)
Panel A: Phase 1 dwell-time PI, ATR(+) Phase-1-only n=3 vs ATR(−) n=4, planned one-sided Mann-Whitney U test comparing ATR(+) > ATR(−).
Panel B: Extinction (Fly 4) dwell-time PI change.  
Panel C: Reversal (Flies 5 & 6) dwell-time PI change.


In [ ]:
# ── Figure 3: Publication-quality combined figure (dwell-time PI) ────────────
# Cohorts: ATR(−) F1–F4 (n=4), ATR(+) Phase-1-only F1–F3 (n=3)
# All panels use centroid dwell-time PI from compute_pi_time()

fig = plt.figure(figsize=(12, 5))
gs  = gridspec.GridSpec(1, 3, width_ratios=[1.4, 1, 1], wspace=0.45)

ATR_NEG_COLOR = '#2166ac'
ATR_POS_COLOR = '#d6604d'
EXT_COLOR     = '#f4a582'
REV_COLOR     = '#762a83'

# ── Panel A: Phase 1 dwell-time PI ──
ax_a = fig.add_subplot(gs[0])

pi_neg_vals = [compute_pi_time(s, ARM_CONFIG)[0]
               for s in ['ATR- F1', 'ATR- F2', 'ATR- F3', 'ATR- F4']]
pi_pos_vals = [compute_pi_time(s, ARM_CONFIG)[0]
               for s in ['ATR+ F1', 'ATR+ F2', 'ATR+ F3']]
pi_neg_vals = [v for v in pi_neg_vals if not np.isnan(v)]
pi_pos_vals = [v for v in pi_pos_vals if not np.isnan(v)]

np.random.seed(0)
for xi, (vals, col) in enumerate([(pi_neg_vals, ATR_NEG_COLOR), (pi_pos_vals, ATR_POS_COLOR)]):
    jit = np.random.uniform(-0.09, 0.09, len(vals))
    ax_a.scatter(xi + jit, vals, color=col, s=65, zorder=4,
                 edgecolors='white', linewidths=0.5, alpha=0.9)
    mn  = np.mean(vals)
    sem = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0
    ax_a.errorbar(xi, mn, yerr=sem, fmt='none', color=col, capsize=5, lw=2, zorder=5)
    ax_a.plot([xi-0.18, xi+0.18], [mn]*2, color=col, lw=2.5, zorder=5)

ax_a.axhline(0, color='#333', lw=1, ls='--', alpha=0.6)
ax_a.set_xticks([0, 1])
ax_a.set_xticklabels([f'ATR(−)\n(n={len(pi_neg_vals)})', f'ATR(+)\n(n={len(pi_pos_vals)})'])
ax_a.set_ylabel('Preference Index (dwell time)', fontsize=10)
ax_a.set_title('A   Phase 1 — Place Conditioning', fontsize=10, fontweight='bold', loc='left')
ax_a.set_ylim(-1.05, 1.05)
ax_a.set_xlim(-0.5, 1.5)
ax_a.yaxis.grid(True, lw=0.4, alpha=0.35)

if pi_neg_vals and pi_pos_vals:
    stat, pval = stats.mannwhitneyu(pi_pos_vals, pi_neg_vals, alternative='greater')
    r_rb = (2 * stat) / (len(pi_pos_vals) * len(pi_neg_vals)) - 1
    sig  = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else f'p={pval:.3f}'
    y_top = max(pi_pos_vals + pi_neg_vals) + 0.12
    ax_a.plot([0, 0, 1, 1], [y_top-0.05, y_top, y_top, y_top-0.05], color='black', lw=1)
    ax_a.text(0.5, y_top+0.02, sig, ha='center', fontsize=10)
    ax_a.text(0.5, 0.02, f'U={stat:.0f}, p={pval:.3f}, r={r_rb:+.2f}',
              transform=ax_a.transAxes, ha='center', fontsize=7.5,
              bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4))

# ── Panel B: Extinction (Fly 4) ──
ax_b = fig.add_subplot(gs[1])
stim4 = compute_pi_time('ATR+ F4s', ARM_CONFIG)[0]
ext4  = compute_pi_time('ATR+ F4e', ARM_CONFIG)[0]
ax_b.plot([0, 1], [stim4, ext4], 'o-', color=EXT_COLOR, lw=2, ms=9,
          markeredgecolor='white', markeredgewidth=0.5)
ax_b.axhline(0, color='#333', lw=1, ls='--', alpha=0.6)
ax_b.set_xticks([0, 1])
ax_b.set_xticklabels(['Stim ON', 'Extinction'])
ax_b.set_ylabel('Preference Index (dwell time)', fontsize=10)
ax_b.set_title('B   Extinction (Fly 4)', fontsize=10, fontweight='bold', loc='left')
ax_b.set_ylim(-1.05, 1.05)
ax_b.yaxis.grid(True, lw=0.4, alpha=0.35)

# ── Panel C: Reversal (Flies 5 & 6) ──
ax_c = fig.add_subplot(gs[2])
for col, (s_id, r_id), lbl in zip(
    ['#762a83', '#1b7837'],
    [('ATR+ F5s', 'ATR+ F5r'), ('ATR+ F6s', 'ATR+ F6r')],
    ['Fly 5', 'Fly 6']
):
    ax_c.plot([0, 1],
              [compute_pi_time(s_id, ARM_CONFIG)[0], compute_pi_time(r_id, ARM_CONFIG)[0]],
              'o-', color=col, lw=2, ms=9,
              markeredgecolor='white', markeredgewidth=0.5, label=lbl)

ax_c.axhline(0, color='#333', lw=1, ls='--', alpha=0.6)
ax_c.set_xticks([0, 1])
ax_c.set_xticklabels(['Stim ON', 'Reversal'])
ax_c.set_ylabel('Preference Index (dwell time)', fontsize=10)
ax_c.set_title('C   Reversal (Flies 5 & 6)', fontsize=10, fontweight='bold', loc='left')
ax_c.set_ylim(-1.05, 1.05)
ax_c.yaxis.grid(True, lw=0.4, alpha=0.35)
ax_c.legend(fontsize=8, frameon=False)

plt.suptitle(
    'Figure 3 — Optogenetic Sweet Neuron Activation: Summary Statistics',
    fontsize=11, fontweight='bold', y=1.02
)
plt.savefig(OUT / 'fig3_summary_publication.pdf', bbox_inches='tight')
plt.savefig(OUT / 'fig3_summary_publication.png', bbox_inches='tight', dpi=200)
plt.show()
print('Saved fig3_summary_publication')


---
## Centroid Dwell-Time PI — Phase 1 Group Comparison
Full centroid-based PI mirrors Figure 2 (entry-count) using continuous position data.  
Cohorts: ATR(−) F1–F4 (n=4) and ATR(+) Phase-1-only F1–F3 (n=3) per report.  
Planned one-sided Mann-Whitney U test comparing ATR(+) > ATR(−); Wilcoxon signed-rank vs 0 for ATR(+).


In [ ]:
# ── Centroid Dwell-Time PI — Phase 1 Group Comparison ─────────────────────────
# Cohorts: ATR(−) F1–F4 (n=4), ATR(+) Phase-1-only F1–F3 (n=3)
# Target values from report:
#   ATR(+) +0.34 ± 0.26;  ATR(−) −0.22 ± 0.32
#   Mann-Whitney U=11.0, p=0.057, r=+0.83
#   Wilcoxon ATR(+) vs 0: W=6, p=0.125

ATR_NEG_COLOR = '#2166ac'
ATR_POS_COLOR = '#d6604d'

pi_time_neg = []
pi_time_pos = []
for sid in ['ATR- F1', 'ATR- F2', 'ATR- F3', 'ATR- F4']:
    pi, _, _ = compute_pi_time(sid, ARM_CONFIG)
    if not np.isnan(pi):
        pi_time_neg.append(pi)
for sid in ['ATR+ F1', 'ATR+ F2', 'ATR+ F3']:
    pi, _, _ = compute_pi_time(sid, ARM_CONFIG)
    if not np.isnan(pi):
        pi_time_pos.append(pi)

print(f'ATR(−) dwell PI: {np.mean(pi_time_neg):+.2f} ± {np.std(pi_time_neg, ddof=1):.2f}  (n={len(pi_time_neg)})')
print(f'ATR(+) dwell PI: {np.mean(pi_time_pos):+.2f} ± {np.std(pi_time_pos, ddof=1):.2f}  (n={len(pi_time_pos)})')

if pi_time_neg and pi_time_pos:
    stat_mw, p_mw = stats.mannwhitneyu(pi_time_pos, pi_time_neg, alternative='greater')
    r_rb = (2 * stat_mw) / (len(pi_time_pos) * len(pi_time_neg)) - 1
    print(f'Mann-Whitney U={stat_mw:.0f}, p={p_mw:.3f}, r={r_rb:+.2f}')

if pi_time_pos:
    stat_w, p_w = stats.wilcoxon(pi_time_pos, alternative='greater')
    print(f'Wilcoxon ATR(+) vs 0: W={stat_w:.0f}, p={p_w:.3f}')

fig, ax = plt.subplots(figsize=(4.5, 5))
np.random.seed(42)
for xi, (vals, col) in enumerate([(pi_time_neg, ATR_NEG_COLOR), (pi_time_pos, ATR_POS_COLOR)]):
    if not vals:
        continue
    jit = np.random.uniform(-0.09, 0.09, len(vals))
    ax.scatter(xi + jit, vals, color=col, s=65, zorder=4, edgecolors='white', linewidths=0.5)
    mn  = np.mean(vals)
    sem = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0
    ax.errorbar(xi, mn, yerr=sem, fmt='none', color=col, capsize=5, lw=2)
    ax.plot([xi-0.18, xi+0.18], [mn]*2, color=col, lw=2.5)

ax.axhline(0, color='#333', lw=1, ls='--', alpha=0.6)
ax.set_xticks([0, 1])
ax.set_xticklabels([f'ATR(−)\n(n={len(pi_time_neg)})', f'ATR(+)\n(n={len(pi_time_pos)})'])
ax.set_ylabel('Preference Index (centroid dwell time)', fontsize=10)
ax.set_title('Phase 1 PI — Centroid Dwell Time', fontsize=11, fontweight='bold')
ax.set_ylim(-1.05, 1.05)
ax.yaxis.grid(True, lw=0.4, alpha=0.35)
plt.tight_layout()
plt.savefig(OUT / 'fig2b_phase1_PI_time.pdf', bbox_inches='tight')
plt.savefig(OUT / 'fig2b_phase1_PI_time.png', bbox_inches='tight', dpi=150)
plt.show()


---
## Figure 4 — Rolling PI Dynamics and Model Fitting
60-second sliding-window PI for each ATR(+) Phase-1-only fly (F1–F3).  
Four candidate models are fit to each fly's rolling-PI timecourse:
1. **Sigmoidal** — ramp from baseline to plateau  
2. **Gaussian** — transient peak then decay  
3. **Linear** — constant drift  
4. **Exponential saturating** — fast rise then plateau  

Best model selected by highest R² (goodness of fit).  
**Report results:** Fly 1 Gaussian R²=0.64; Fly 2 Gaussian R²=0.98; Fly 3 Sigmoidal R²=0.93.


In [ ]:
# ── Figure 4: Rolling PI Model Fitting ────────────────────────────────────────────
# 60-second sliding window PI for ATR(+) Phase-1-only flies (n=3).
# Report: Fly1 Gaussian R2=0.64; Fly2 Gaussian R2=0.98; Fly3 Sigmoidal R2=0.93

from scipy.optimize import curve_fit

WINDOW_S  = 60
WINDOW_FR = WINDOW_S * FPS
STEP_FR   = 1

def rolling_pi_series(sid, arm_cfg):
    cfg     = arm_cfg.get(sid, {})
    x_bound = cfg.get('x_boundary')
    r_side  = cfg.get('reward_side')
    if x_bound is None or r_side is None:
        return None, None
    xv = DATA[sid]['x']
    n  = len(xv)
    t_centers, pis = [], []
    for start in range(0, n - WINDOW_FR + 1, STEP_FR):
        end = start + WINDOW_FR
        win = xv[start:end]
        valid = ~np.isnan(win)
        if valid.sum() < WINDOW_FR * 0.5:
            continue
        xw = win[valid]
        if r_side == 'right':
            nr, nu = np.sum(xw > x_bound), np.sum(xw < x_bound)
        else:
            nr, nu = np.sum(xw < x_bound), np.sum(xw > x_bound)
        total = nr + nu
        t_centers.append((start + WINDOW_FR / 2) / FPS / 60)
        pis.append((nr - nu) / total if total > 0 else np.nan)
    return np.array(t_centers), np.array(pis)

def sigmoid(t, L, k, t0, b):       return L / (1 + np.exp(-k * (t - t0))) + b
def gaussian_m(t, A, mu, sig, b):  return A * np.exp(-0.5 * ((t - mu) / sig)**2) + b
def linear_m(t, m, b):             return m * t + b
def exp_sat(t, A, tau, b):         return A * (1 - np.exp(-(t - t[0]) / tau)) + b

MODELS = {
    'sigmoidal':      (sigmoid,     [1.0, 1.0, 5.0, 0.0]),
    'Gaussian':       (gaussian_m,  [1.0, 5.0, 2.0, 0.0]),
    'linear':         (linear_m,    [0.05, 0.0]),
    'exp_saturating': (exp_sat,     [1.0, 2.0, 0.0]),
}

def fit_best_model(t, pi):
    best_name, best_r2, best_yfit = 'none', -np.inf, np.zeros_like(pi)
    for name, (func, p0) in MODELS.items():
        try:
            popt, _ = curve_fit(func, t, pi, p0=p0, maxfev=10000)
            yfit = func(t, *popt)
            ss_res = np.sum((pi - yfit)**2)
            ss_tot = np.sum((pi - np.mean(pi))**2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
            if r2 > best_r2:
                best_r2, best_name, best_yfit = r2, name, yfit
        except Exception:
            pass
    return best_name, best_r2, best_yfit

phase1_pos = ['ATR+ F1', 'ATR+ F2', 'ATR+ F3']
colors_p1  = ['#e31a1c', '#ff7f00', '#33a02c']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
fig.suptitle(
    'Figure 4 — Rolling PI Dynamics (60-s window) with Best-Fit Model',
    fontsize=12, fontweight='bold'
)

for ax, sid, col in zip(axes, phase1_pos, colors_p1):
    t_arr, pi_arr = rolling_pi_series(sid, ARM_CONFIG)
    if t_arr is None:
        ax.text(0.5, 0.5, 'ARM_CONFIG not set', transform=ax.transAxes,
                ha='center', va='center', color='gray')
        ax.set_title(sid)
        continue

    ok = ~np.isnan(pi_arr)
    t_ok, pi_ok = t_arr[ok], pi_arr[ok]

    best_name, best_r2, best_fit = fit_best_model(t_ok, pi_ok)

    ax.plot(t_ok, pi_ok, color=col, lw=1.0, alpha=0.45, label='Rolling PI')
    ax.plot(t_ok, best_fit, color='black', lw=2.0, ls='--',
            label=f'{best_name} (R²={best_r2:.2f})')
    ax.axhline(0, color='gray', lw=0.8, ls=':', alpha=0.7)
    ax.set_title(DATA[sid]['label'].replace('\n', ' '), fontsize=10, fontweight='bold')
    ax.set_xlabel('Time (min)', fontsize=9)
    ax.set_ylabel('Preference Index (60-s window)', fontsize=9)
    ax.set_ylim(-1.05, 1.05)
    ax.yaxis.grid(True, lw=0.4, alpha=0.35)
    ax.legend(fontsize=7.5, frameon=False)
    print(f"{sid}: best model = {best_name:20s}  R² = {best_r2:.3f}")

plt.tight_layout()
FIGS = OUT / 'figures'
FIGS.mkdir(exist_ok=True)
plt.savefig(FIGS / 'fig4_rolling_PI_model_fit.png', dpi=150, bbox_inches='tight')
plt.savefig(FIGS / 'fig4_rolling_PI_model_fit.pdf', bbox_inches='tight')
plt.show()
print('Saved Figure 4 -> figures/fig4_rolling_PI_model_fit.png')


---
## Figure 5 — Movement Speed: Phase 1 (Reward vs. Non-Reward Arm)
Violin plots of per-frame centroid speed (px/sec), filtered to the 85th percentile,
split by arm assignment.

**Cohort:** ATR(−) controls F1–F4 (n=4) and ATR(+) Phase-1-only F1–F3 (n=3).

**Arm labeling for speed (REWARD\_SIDES\_P1):**  
The speed figure uses `reward_side = 'right'` for all Phase 1 flies (ATR+ and ATR−).
This reproduces the speed values reported in the lab report (Fig 5):  
- ATR+ F1: reward arm ≈65 px/s vs non-reward ≈43 px/s  
- ATR+ F2: reward arm ≈60 px/s vs non-reward ≈46 px/s  
- ATR+ F3: similar speeds (≈65 vs ≈63 px/s)  

The reward-arm boundary used here is auto-detected from the X distribution
(bimodal valley), which may differ slightly from the ARM\_CONFIG x\_boundary.


In [ ]:
# ── Speed Analysis Helper Functions ──────────────────────────────────────────
from scipy.signal import find_peaks
from scipy.ndimage import uniform_filter1d
from matplotlib.patches import Patch

def compute_speed(x, y):
    """Frame-to-frame movement speed in pixels/sec."""
    dx = np.diff(np.where(np.isnan(x), np.nan, x))
    dy = np.diff(np.where(np.isnan(y), np.nan, y))
    return np.sqrt(dx**2 + dy**2) * FPS

def detect_arm_boundary(xv):
    """Auto-detect X boundary between the two arms from bimodal distribution."""
    counts, edges = np.histogram(xv, bins=100)
    centers = (edges[:-1] + edges[1:]) / 2
    sm = uniform_filter1d(counts.astype(float), size=5)
    peaks, _ = find_peaks(sm, distance=10, prominence=sm.max() * 0.08)
    if len(peaks) >= 2:
        p1, p2 = sorted(peaks[np.argsort(sm[peaks])[-2:]])
        valley = p1 + np.argmin(sm[p1:p2 + 1])
        return float(centers[valley])
    return float(np.median(xv))

def split_arm_speed(sid, reward_side='right', pct=85):
    """Return (reward_speeds, non-reward_speeds), each filtered to `pct` percentile."""
    d = DATA[sid]
    boundary = detect_arm_boundary(d['xv'])
    spd = compute_speed(d['x'], d['y'])
    xp  = d['x'][:-1]
    ok  = ~(np.isnan(spd) | np.isnan(xp))
    spd, xp = spd[ok], xp[ok]
    in_rew = (xp > boundary) if reward_side == 'right' else (xp < boundary)
    def filt(a): return a[a <= np.percentile(a, pct)] if len(a) else a
    return filt(spd[in_rew]), filt(spd[~in_rew]), boundary

def draw_violin(ax, data, x, color, width=0.38, alpha=0.85):
    if len(data) < 5:
        return
    vp = ax.violinplot(data, positions=[x], widths=[width],
                       showmeans=False, showmedians=False, showextrema=False)
    for b in vp['bodies']:
        b.set_facecolor(color); b.set_edgecolor(color); b.set_alpha(alpha)
    ax.plot([x - width * 0.3, x + width * 0.3], [np.mean(data)] * 2,
            color='black', lw=2, solid_capstyle='butt')

# ── Figure 5: Phase 1 Movement Speed by Arm ──────────────────────────────────
# All Phase 1 flies use reward_side='right' (high-X arm labeled as reward).
# This reproduces the speed values from the lab report:
#   ATR+ F1: 65 vs 43 px/s | ATR+ F2: 60 vs 46 px/s | ATR+ F3: 65 vs 63 px/s
# Note: ARM_CONFIG uses 'left' for ATR+ F1-F3 to compute PI (different metric).
# The speed figure and PI figure each use the arm convention that matches the
# corresponding reported values in the lab report.
REWARD_SIDES_P1 = {
    'ATR- F1': 'right', 'ATR- F2': 'right', 'ATR- F3': 'right', 'ATR- F4': 'right',
    'ATR+ F1': 'right', 'ATR+ F2': 'right', 'ATR+ F3': 'right',
}

NEG_R = '#4393c3'   # blue   — ATR(−) reward arm
NEG_N = '#f4a582'   # salmon — ATR(−) non-reward arm
POS_R = '#66c2a5'   # teal   — ATR(+) reward arm
POS_N = '#fdae6b'   # orange — ATR(+) non-reward arm

fly_order_p1 = ['ATR- F1', 'ATR- F2', 'ATR- F3', 'ATR- F4',
                'ATR+ F1', 'ATR+ F2', 'ATR+ F3']

fig, ax = plt.subplots(figsize=(16, 6))
xt, xl, px, sep_x = [], [], 0, None

for sid in fly_order_p1:
    if sid not in DATA:
        px += 1.0
        continue
    is_neg = DATA[sid]['group'] == 'neg'
    rc, nc = (NEG_R, NEG_N) if is_neg else (POS_R, POS_N)
    s_r, s_n, _ = split_arm_speed(sid, REWARD_SIDES_P1[sid])
    draw_violin(ax, s_r, px - 0.22, rc)
    draw_violin(ax, s_n, px + 0.22, nc)
    xt.append(px)
    xl.append(sid)
    if sid == 'ATR- F4':
        sep_x = px + 0.75
        ax.axvline(sep_x, color='gray', lw=1, ls='--', alpha=0.7)
        px += 1.5
    else:
        px += 1.0

if sep_x is not None:
    ax.text(sep_x - 0.1, 0.97, 'ATR−', transform=ax.get_xaxis_transform(),
            color=NEG_R, ha='right', va='top', fontsize=9, fontweight='bold')
    ax.text(sep_x + 0.1, 0.97, 'ATR+', transform=ax.get_xaxis_transform(),
            color=POS_R, ha='left', va='top', fontsize=9, fontweight='bold')

ax.set_xticks(xt)
ax.set_xticklabels(xl, fontsize=9)
ax.set_ylabel('Movement Speed (pixels/sec)', fontsize=10)
ax.set_title(
    'Figure 5 — Phase 1 Movement Speed: Reward vs. Non-Reward Arm\n'
    'ATR(−) F1–4 (n=4) and ATR(+) Phase-1-only F1–3 (n=3) | 85th-pct filtered',
    fontsize=12, fontweight='bold')
ax.yaxis.grid(True, lw=0.4, alpha=0.35)

legend_elems = [
    Patch(facecolor=NEG_R, label='ATR− Reward arm'),
    Patch(facecolor=NEG_N, label='ATR− Non-reward arm'),
    Patch(facecolor=POS_R, label='ATR+ Reward arm'),
    Patch(facecolor=POS_N, label='ATR+ Non-reward arm'),
]
ax.legend(handles=legend_elems, fontsize=8, frameon=False, loc='upper right')
plt.tight_layout()

FIGS = OUT / 'figures'
FIGS.mkdir(exist_ok=True)
plt.savefig(FIGS / 'fig5_phase1_speed.png', dpi=120, bbox_inches='tight')
plt.savefig(FIGS / 'fig5_phase1_speed.pdf', bbox_inches='tight')
plt.show()
print("Saved Figure 5 -> figures/fig5_phase1_speed.png")

---
## Figure 6 — Phase 2 Movement Speed: Extinction & Reversal
Same per-frame speed analysis for the three Phase 2 flies (Fly 4 extinction,
Flies 5 & 6 reversal). Top row = Stim ON phase; bottom row = Phase 2 phase.

**Arm sides** match PHASE2\_CFG, which is consistent with ARM\_CONFIG:  
- Fly 4 Stim: reward = RIGHT (high X); Fly 4 Extinction: old reward = LEFT (low X)  
- Fly 5 Stim: reward = LEFT (low X);  Fly 5 Reversal: old reward = LEFT (new reward = RIGHT)  
- Fly 6 Stim: reward = LEFT (low X);  Fly 6 Reversal: old reward = LEFT (new reward = RIGHT)  

Δ = mean(reward arm speed) − mean(non-reward arm speed), in px/sec.


In [ ]:
# ── Figure 6: Phase 2 Movement Speed — Extinction & Reversal ─────────────────
# Arm assignments match Figure 1 of the lab report (REW label positions):
#   F4 stim: reward RIGHT  | F4 extinction: old reward LEFT
#   F5 stim: reward LEFT   | F5 reversal: old reward LEFT  (new reward RIGHT)
#   F6 stim: reward LEFT   | F6 reversal: old reward RIGHT (new reward LEFT)

PHASE2_CFG = {
    'stim': {
        'ATR+ F4s': {'side': 'right', 'rew_lbl': 'Reward\n(RIGHT)', 'non_lbl': 'Non-reward', 'fly': 'Fly 4'},
        'ATR+ F5s': {'side': 'left',  'rew_lbl': 'Reward\n(LEFT)',  'non_lbl': 'Non-reward', 'fly': 'Fly 5'},
        'ATR+ F6s': {'side': 'left',  'rew_lbl': 'Reward\n(LEFT)',  'non_lbl': 'Non-reward', 'fly': 'Fly 6'},
    },
    'p2': {
        'ATR+ F4e': {
            'side': 'left',  'type': 'extinction', 'title': 'Fly 4 — Extinction',
            'rew_lbl': 'Old reward\n(LEFT)', 'non_lbl': 'Other arm',
        },
        'ATR+ F5r': {
            'side': 'left',  'type': 'reversal',   'title': 'Fly 5 — Reversal',
            'rew_lbl': 'Old reward\n(LEFT)', 'non_lbl': 'NEW reward\n(RIGHT)',
        },
        'ATR+ F6r': {
            'side': 'right', 'type': 'reversal',   'title': 'Fly 6 — Reversal',
            'rew_lbl': 'Old reward\n(RIGHT)', 'non_lbl': 'NEW reward\n(LEFT)',
        },
    },
}

STIM_R = '#66c2a5'   # green  — reward arm during stim
STIM_N = '#d9d9d9'   # gray   — non-reward arm
OLD_R  = '#f4a582'   # salmon — old reward arm (phase 2)
NEW_R  = '#9970ab'   # purple — new reward arm (reversal)

stim_ids = ['ATR+ F4s', 'ATR+ F5s', 'ATR+ F6s']
p2_ids   = ['ATR+ F4e', 'ATR+ F5r', 'ATR+ F6r']

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey='row')
fig.suptitle(
    'Figure 6 — Phase 2 Movement Speed: Extinction & Reversal Flies\n'
    'Stim ON (top) vs. Phase 2 (bottom) | mean ± SEM, 85th-pct filtered',
    fontsize=12, fontweight='bold')

for col, (stim_id, p2_id) in enumerate(zip(stim_ids, p2_ids)):
    cfg_s = PHASE2_CFG['stim'][stim_id]
    cfg_p = PHASE2_CFG['p2'][p2_id]

    # ── Top: Stim ON ──────────────────────────────────────────────────────────
    ax = axes[0, col]
    s_r, s_n, _ = split_arm_speed(stim_id, cfg_s['side'])
    draw_violin(ax, s_r, 0, STIM_R, width=0.6)
    draw_violin(ax, s_n, 1, STIM_N, width=0.6)
    delta = np.mean(s_r) - np.mean(s_n) if len(s_r) and len(s_n) else np.nan
    ax.set_title(f"{cfg_s['fly']} — Stim ON", fontsize=10, fontweight='bold', color='#1a7a4a')
    ax.text(0.5, 0.97, f'Δ = {delta:.1f} px/s', transform=ax.transAxes,
            ha='center', va='top', fontsize=8)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([cfg_s['rew_lbl'], cfg_s['non_lbl']], fontsize=8)
    if col == 0: ax.set_ylabel('Movement Speed (pixels/sec)', fontsize=9)
    ax.yaxis.grid(True, lw=0.4, alpha=0.35)

    # ── Bottom: Phase 2 ───────────────────────────────────────────────────────
    ax = axes[1, col]
    s_r, s_n, _ = split_arm_speed(p2_id, cfg_p['side'])
    v2_color = STIM_N if cfg_p['type'] == 'extinction' else NEW_R
    draw_violin(ax, s_r, 0, OLD_R, width=0.6)
    draw_violin(ax, s_n, 1, v2_color, width=0.6)
    delta = np.mean(s_r) - np.mean(s_n) if len(s_r) and len(s_n) else np.nan
    ax.set_title(cfg_p['title'], fontsize=10, fontweight='bold', color='#cc4c02')
    ax.text(0.5, 0.97, f'Δ = {delta:.1f} px/s', transform=ax.transAxes,
            ha='center', va='top', fontsize=8)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([cfg_p['rew_lbl'], cfg_p['non_lbl']], fontsize=8)
    if col == 0: ax.set_ylabel('Movement Speed (pixels/sec)', fontsize=9)
    ax.yaxis.grid(True, lw=0.4, alpha=0.35)

legend_elems = [
    Patch(facecolor=STIM_R, label='Original reward arm (Stim ON)'),
    Patch(facecolor=STIM_N, label='Non-reward / other arm'),
    Patch(facecolor=OLD_R,  label='Old reward arm (Phase 2)'),
    Patch(facecolor=NEW_R,  label='New reward arm (Reversal)'),
]
fig.legend(handles=legend_elems, fontsize=8, frameon=False,
           loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=[0, 0.05, 1, 1])

FIGS = OUT / 'figures'
FIGS.mkdir(exist_ok=True)
plt.savefig(FIGS / 'fig6_phase2_speed.png', dpi=120, bbox_inches='tight')
plt.savefig(FIGS / 'fig6_phase2_speed.pdf', bbox_inches='tight')
plt.show()
print("Saved Figure 6 -> figures/fig6_phase2_speed.png")